In [1]:

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from bootcampviztools import plot_combined_graphs, pinta_distribucion_categoricas, plot_categorical_relationship_fin,\
      plot_grouped_boxplots,plot_grouped_histograms, grafico_dispersion_con_correlacion, plot_categorical_numerical_relationship, bubble_plot
from scipy.stats import pearsonr, chi2_contingency, mannwhitneyu,f_oneway

Cargo los datos inmobiliarios

In [2]:
df_inmnu = pd.read_csv("C:/Users/Ariela/Desktop/EDA - Inmobiliario/Inmobiliario/src/data/EDA_Inmob_Nunoa.csv")
df_inmlb = pd.read_csv("C:/Users/Ariela/Desktop/EDA - Inmobiliario/Inmobiliario/src/data/EDA_Inmob_Lascondes.csv")
df_inmlc = pd.read_csv("C:/Users/Ariela/Desktop/EDA - Inmobiliario/Inmobiliario/src/data/EDA_Inmob_Lobarnechea.csv")

In [3]:
df_inmnu["comuna"] = "Ñuñoa"
df_inmlb["comuna"] = "Las Condes"
df_inmlc["comuna"] = "Lo Barnechea"

df_total = pd.concat([df_inmnu, df_inmlb, df_inmlc], axis=0).reset_index(drop=True)
df_total = df_total.dropna()

#Corrijo los valores
df_total['precio'] = pd.to_numeric(df_total['precio'].str.replace('$', '', regex=False).str.replace('.', '', regex=False).str.strip())  #Voy a corregir la columna de precio, porque piensa que es un str. Elimino el signo $ y los puntos
df_total['m2'] = pd.to_numeric(df_total['m2'].str.replace(" m²", "", regex=False).str.replace(",", ".", regex=False))
df_total['precio_euro'] = df_total['precio']*0.00094
df_total['precio_eurom2'] = df_total['precio_euro']/df_total['m2']
df_total[["comuna", "m2", "precio_eurom2"]].head(10)

# Limpio los outlayers
p95 = df_total["precio_eurom2"].quantile(0.95)
p05 = df_total["precio_eurom2"].quantile(0.05)


df_total = df_total[(df_total["precio_eurom2"] > p05) & (df_total["precio_eurom2"] < p95)]


In [5]:
# Test anova de precios/m2

from scipy.stats import f_oneway

nunoa = df_total[df_total["comuna"] == "Ñuñoa"]["precio_eurom2"].dropna()
lascondes = df_total[df_total["comuna"] == "Las Condes"]["precio_eurom2"].dropna()
lobarnechea = df_total[df_total["comuna"] == "Lo Barnechea"]["precio_eurom2"].dropna()

stat, p_valor = f_oneway(nunoa, lascondes, lobarnechea)
print(f"Estadístico F: {stat:.4f}")
print(f"P-valor: {p_valor:.4f}")
print(f"P-valor: {p_valor:.2e}")

Estadístico F: 62.2101
P-valor: 0.0000
P-valor: 4.00e-25


In [13]:
print(df_total.groupby("comuna")["precio_eurom2"].describe())

              count       mean         std       min        25%        50%  \
comuna                                                                       
Las Condes    258.0  30.902302  124.160571  0.000000  12.236001  15.167292   
Lo Barnechea   46.0  15.000849    7.667578  0.000231  10.841142  13.130483   
Ñuñoa         323.0  25.434811  184.018791  0.000282   9.641184  11.280000   

                    75%          max  
comuna                                
Las Condes    20.197005  1745.816880  
Lo Barnechea  16.447604    41.777778  
Ñuñoa         13.227742  3300.444444  


In [23]:
# Test anova de precios

from scipy.stats import f_oneway

nunoa = df_total[df_total["comuna"] == "Ñuñoa"]["precio_euro"].dropna()
lascondes = df_total[df_total["comuna"] == "Las Condes"]["precio_euro"].dropna()
lobarnechea = df_total[df_total["comuna"] == "Lo Barnechea"]["precio_euro"].dropna()

stat, p_valor = f_oneway(nunoa, lascondes, lobarnechea)
print(f"Estadístico F: {stat:.4f}")
print(f"P-valor: {p_valor:.4f}")

Estadístico F: 1.1012
P-valor: 0.3331


In [21]:
print(df_total.groupby("comuna")["precio_euro"].describe())


              count         mean          std      min          25%  \
comuna                                                                
Las Condes    258.0  1748.895948  1029.133320  0.00000  1095.154050   
Lo Barnechea   46.0  1885.382563  1475.842898  0.07520   937.210315   
Ñuñoa         323.0  1065.460048  8234.373746  0.01974   460.624910   

                     50%          75%          max  
comuna                                              
Las Condes    1535.50034  2068.000000    6580.0000  
Lo Barnechea  1578.32251  2317.294345    9362.8042  
Ñuñoa          545.20000   658.000000  148520.0000  
